In [1]:
import numpy as np
import pyhdf
from pyhdf.SD import SD, SDC
from pyhdf.HDF import *
from pyhdf.VS import *
import glob
import os
import netCDF4 as nc
import sys

'''
CONFIGURATION PARAMETERS
'''

LATLONDATA = "/home/al8425b-hpc/NASA/cropTest/testData/ABI_EAST_GEO_TOPO_LOMSK.nc"
CLOUDSATPATH = '/home/al8425b-hpc/NASA/cropTest/testData/cloudsat/'
ROOT_DIR = '/home/al8425b-hpc/NASA/cropTest/testData/cloudsat/2B-CLDCLASS-LIDAR'
ABIDATA = "/home/al8425b-hpc/NASA/cropTest/testData/GOES_Data/" 
# SAVEDIR = '/home/al8425b-hpc/NASA/satvision-pix4d/examples/originalChipTest/output/'
SAVEDIR = '/home/al8425b-hpc/NASA/satvision-pix4d/examples/abi_3d_reconstruction/originalChipTest/output'

In [2]:
import os
import glob

# Test targets
test_year = "2020"
test_day = "001"

# Let's look inside the directory to see what orbits exist
sample_dir = os.path.join(ROOT_DIR, test_year, test_day)
try:
    print("Available files in this directory:")
    for f in os.listdir(sample_dir):
        if f.endswith('hdf'):
            print(f)
except FileNotFoundError:
    print(f"Directory not found: {sample_dir}. Check your paths or try a different year/day.")

Available files in this directory:
2020001002545_72860_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E09_F00.hdf
2020001020417_72861_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E09_F00.hdf
2020001065952_72864_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E09_F00.hdf
2020001083824_72865_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E09_F00.hdf
2020001101655_72866_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E09_F00.hdf
2020001115527_72867_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E09_F00.hdf
2020001133359_72868_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E09_F00.hdf
2020001151230_72869_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E09_F00.hdf
2020001165102_72870_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E09_F00.hdf
2020001182934_72871_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E09_F00.hdf
2020001200806_72872_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E09_F00.hdf
2020001214637_72873_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E09_F00.hdf
2020001232509_72874_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E09_F00.hdf


In [3]:
test_year = '2020'
test_day = '001'
test_orbit = '20200'

In [4]:

# Multi-timestep configuration
OFFSETS_MINUTES = [-40, -20, 0, 20, 40]   # every 20 min centered on CloudSat time
CHIP_HALF_SIZE = 64                        # 128x128 chips
ALLOW_MISSING_TIMESTEPS = False            # set True if you want to allow partial time series

## Core Utilities
Grouping the "helper" functions together. These are the mathematical or logical tools that don't directly open files but manipulate data in the background.

In [5]:

def is_leap_year(year):
    year = int(year)
    return (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)

def increment_day(year, day, step):
    year = int(year)
    day = int(day)

    while step != 0:
        days_in_year = 366 if is_leap_year(year) else 365

        if step > 0:
            day += 1
            if day > days_in_year:
                year += 1
                day = 1
            step -= 1
        else:
            day -= 1
            if day < 1:
                year -= 1
                day = 366 if is_leap_year(year) else 365
            step += 1

    return str(year), f"{day:03d}"

def shift_time(yy, ddn, t_hours, delta_minutes):
    total_minutes = t_hours * 60.0 + delta_minutes
    day_shift = 0

    while total_minutes < 0:
        total_minutes += 24 * 60
        day_shift -= 1

    while total_minutes >= 24 * 60:
        total_minutes -= 24 * 60
        day_shift += 1

    new_t_hours = total_minutes / 60.0
    new_yy, new_ddn = increment_day(yy, ddn, day_shift)

    return new_yy, new_ddn, new_t_hours

def interpArray(Temperature, EC_height):
    z_grid = np.arange(40) * 0.5
    N = Temperature.shape[0]
    Temperature_grid = np.zeros((N, len(z_grid)), dtype=np.float32)

    for i in range(N):
        Temperature_tmp = np.squeeze(Temperature[i, :])
        ivalid = np.squeeze(np.where(Temperature_tmp > 0))

        Temperature_grid[i, :] = np.interp(
            z_grid,
            np.flip(EC_height[ivalid] / 1000.),
            np.flip(Temperature_tmp[ivalid])
        )

    return Temperature_grid

## The Data Readers

In [6]:

def read_2b_cldclass_lidar(cs_file, latbin=None):
    f_2b_cldclass_lidar = SD(cs_file, SDC.READ)
    sds_obj = f_2b_cldclass_lidar.select('CloudLayerBase')
    cs_clb = sds_obj.get()
    sds_obj = f_2b_cldclass_lidar.select('CloudLayerTop')
    cs_clt = sds_obj.get()
    sds_obj = f_2b_cldclass_lidar.select('CloudLayerType')
    cs_cltype = sds_obj.get()

    sdc_2bcldclass_lidar = HDF(cs_file, SDC.READ)
    vs_2bcldclass_lidar = sdc_2bcldclass_lidar.vstart()
    cs_QC = np.squeeze(vs_2bcldclass_lidar.attach('Data_quality')[:])
    Latitude = np.squeeze(vs_2bcldclass_lidar.attach('Latitude')[:])
    Longitude = np.squeeze(vs_2bcldclass_lidar.attach('Longitude')[:])

    ilat = np.squeeze(np.argwhere(
        (Latitude >= latbin[0]) & (Latitude <= latbin[1])))
    cs_clb = cs_clb[ilat, :]
    cs_clt = cs_clt[ilat, :]
    cs_QC = cs_QC[ilat]
    cs_cltype = cs_cltype[ilat, :]
    Latitude = Latitude[ilat]
    Longitude = Longitude[ilat]

    return (cs_clb, cs_clt, cs_cltype, cs_QC, Latitude, Longitude)

# ===============================================


def read_cs_ecmwf(aux_file, latbin=None):
    f_ecmwf = SD(aux_file, SDC.READ)
    sds_obj = f_ecmwf.select('Pressure')
    Pressure = sds_obj.get()
    sds_obj = f_ecmwf.select('Temperature')
    Temperature = sds_obj.get()
    sds_obj = f_ecmwf.select('Specific_humidity')
    Specific_humidity = sds_obj.get()

    sdc_ecmwf = HDF(aux_file, SDC.READ)
    vs_ecmwf = sdc_ecmwf.vstart()
    EC_height = np.squeeze(vs_ecmwf.attach('EC_height')[:])
    Profile_time = np.squeeze(vs_ecmwf.attach('Profile_time')[:])
    UTC_start = np.squeeze(vs_ecmwf.attach('UTC_start')[:])
    Latitude = np.squeeze(vs_ecmwf.attach('Latitude')[:])
    Longitude = np.squeeze(vs_ecmwf.attach('Longitude')[:])
    DEM_elevation = np.squeeze(vs_ecmwf.attach('DEM_elevation')[:])
    Skin_temperature = np.squeeze(vs_ecmwf.attach('Skin_temperature')[:])
    Surface_pressure = np.squeeze(vs_ecmwf.attach('Surface_pressure')[:])
    Temperature_2m = np.squeeze(vs_ecmwf.attach('Temperature_2m')[:])
    U10_velocity = np.squeeze(vs_ecmwf.attach('U10_velocity')[:])
    V10_velocity = np.squeeze(vs_ecmwf.attach('V10_velocity')[:])

    UTC_Time = UTC_start + Profile_time
    UTC_Time = UTC_Time / 60. / 60.

    ilat = np.squeeze(np.argwhere(
        (Latitude >= latbin[0]) & (Latitude <= latbin[1])))
    Pressure = Pressure[ilat, :]
    Temperature = Temperature[ilat, :]
    Specific_humidity = Specific_humidity[ilat, :]
    DEM_elevation = DEM_elevation[ilat]
    Skin_temperature = Skin_temperature[ilat]
    Temperature_2m = Temperature_2m[ilat]
    U10_velocity = U10_velocity[ilat]
    V10_velocity = V10_velocity[ilat]
    UTC_Time = UTC_Time[ilat]

    return (Pressure, DEM_elevation, Temperature, Specific_humidity, EC_height,
            Temperature_2m, U10_velocity, V10_velocity, UTC_Time)


In [7]:
BOUND_SIZE = 1600
LENGTH = 10848

f = nc.Dataset(LATLONDATA)
abiLong = np.array(f['Longitude'])
abiLat = np.array(f['Latitude'])

abiLongB = abiLong[BOUND_SIZE:LENGTH-BOUND_SIZE, BOUND_SIZE:LENGTH-BOUND_SIZE]
abiLatB = abiLat[BOUND_SIZE:LENGTH-BOUND_SIZE, BOUND_SIZE:LENGTH-BOUND_SIZE]
abiLongB[abiLongB == -999] = 10
abiLatB[abiLatB == -999] = 10
abiLongB[abiLongB < 0] += 360

longMin = abiLongB.min()
longMax = abiLongB.max()
latMin = abiLatB.min()
latMax = abiLatB.max()

GATHERED_ABI_FILES = {}
COLLECTED_ABI_DATA = {}

latSlice = abiLat[:, 5424]
latSlice = latSlice[18:-18]
longSlice = abiLong[5424, :]
longSlice = longSlice[18:-18]
latSlice = latSlice[::-1]

In [11]:
# 1. Recreate the file lookup path using glob (just like the script does)
cs_file_list = glob.glob(os.path.join(
    CLOUDSATPATH, '2B-CLDCLASS-LIDAR', test_year, test_day,
    test_year + test_day + '*' + test_orbit + '*2B-CLDCLASS-LIDAR*' + 'P1_R05*.hdf'
))
print(cs_file_list)

aux_file_list = glob.glob(os.path.join(
    CLOUDSATPATH, 'ECMWF-AUX', test_year, test_day,
    test_year + test_day + '*' + test_orbit + '*ECMWF-AUX*' + 'P1_R05*.hdf'
))

# # 2. Make sure we actually found the files before reading them
if len(cs_file_list) == 0 or len(aux_file_list) == 0:
    print("🚨 Could not find matching CloudSat or ECMWF files. Double check your orbit number!")
else:
    print(f"Found CloudSat file: {cs_file_list[0]}")
    print(f"Found ECMWF file: {aux_file_list[0]}")
    print("-" * 50)

#     # 3. Define the latitude bounding box (using the variables calculated in the script)
#     lat_bounds = (latMin, latMax)

#     # 4. Run the ECMWF reader and inspect the array shapes
#     print("Reading ECMWF data...")
#     Pressure, DEM_elevation, Temperature, Specific_humidity, EC_height, \
#     Temperature_2m, U10_velocity, V10_velocity, UTC_Time = read_cs_ecmwf(aux_file_list[0], latbin=lat_bounds)
    
#     print(f" -> Temperature profile shape: {Temperature.shape}")
#     print(f" -> Surface Temperature 2m shape: {Temperature_2m.shape}")

#     # 5. Run the CloudSat reader and inspect the array shapes
#     print("\nReading CloudSat data...")
#     cs_clb, cs_clt, cs_cltype, cs_QC, Latitude, Longitude = read_2b_cldclass_lidar(cs_file_list[0], latbin=lat_bounds)
    
#     print(f" -> Cloud Type mask shape: {cs_cltype.shape}")
#     print(f" -> Number of coordinates in track segment: {len(Latitude)}")

[]
🚨 Could not find matching CloudSat or ECMWF files. Double check your orbit number!
